# T2: クリーンロングラン抽出 (R01-R03)

**目的**: R01〜R03のレースデータからロングランを抽出し、クリーンラップのみを特定する。

## ロングラン識別ロジック（CLAUDE.md定義に準拠）
1. アウトラップ（PitOutTime_secが非NaN）を除外
2. インラップ（PitInTime_secが非NaN）を除外
3. 同一ドライバー・同一Stint・同一Compoundの連続ラップをグループ化
4. グループ内ラップ数 >= 5 → ロングラン
5. 各ラップが当該GPセッション最速の107%以内（異常値除外フィルタ）
6. TrackStatus == '1'（グリーンフラグ）のラップのみ

In [ ]:
import matplotlib
matplotlib.use('Agg')  # GUIなし環境用バックエンド

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import os
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 日本語フォント設定
def _set_japanese_font():
    for font in fm.fontManager.ttflist:
        if font.name == 'Hiragino Sans':
            matplotlib.rcParams['font.family'] = font.name
            print(f'日本語フォント: {font.name}')
            return
    print('Hiragino Sans未検出 → DejaVu Sans使用（日本語はグラフに非表示）')

_set_japanese_font()
print('ライブラリ読み込み完了')

In [ ]:
# ==============================
# 設定
# ==============================
BASE_DIR = '/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised'
OUTPUT_DIR = os.path.join(BASE_DIR, 'notebooks', 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GP設定: (ラベル, CSVパス)
GP_FILES = [
    ('R01_Australia', os.path.join(BASE_DIR, 'data', '2026_R01_Australia', 'export', 'race_laps.csv')),
    ('R02_China',     os.path.join(BASE_DIR, 'data', '2026_R02_China',     'export', 'race_laps.csv')),
    ('R03_Japan',     os.path.join(BASE_DIR, 'data', '2026_R03_Japan',     'export', 'race_laps.csv')),
]

LONGRUN_MIN_LAPS = 5   # ロングランの最低ラップ数
FILTER_PCT = 1.07      # セッション最速ラップの107%以内フィルタ
GREEN_FLAG = '1'       # TrackStatus グリーンフラグ値（文字列比較）

STYLE = {
    'bg_color':   '#1a1a2e',
    'text_color': '#ffffff',
    'grid_color': '#333355',
    'figsize':    (12, 6.75),
    'title_size': 16,
    'label_size': 12,
}

COMPOUND_COLORS = {
    'SOFT':         '#FF3333',
    'MEDIUM':       '#FFD700',
    'HARD':         '#FFFFFF',
    'INTERMEDIATE': '#39B54A',
    'WET':          '#0072CE',
}

print(f'出力先: {OUTPUT_DIR}')

## Step 1: データ読み込み

In [ ]:
def load_race_laps(gp_label, csv_path):
    """race_laps.csvを読み込み、型変換して返す"""
    if not os.path.exists(csv_path):
        print(f'[WARNING] {gp_label}: ファイルが見つかりません → {csv_path}')
        return None

    # TrackStatusは文字列として読み込む（数値と文字列の混在を防ぐ）
    df = pd.read_csv(csv_path, dtype={'TrackStatus': str})
    df['GP'] = gp_label

    # 数値カラムの型変換
    for col in ['LapTime_sec', 'Sector1Time_sec', 'Sector2Time_sec', 'Sector3Time_sec',
                'Stint', 'TyreLife', 'LapNumber', 'PitOutTime_sec', 'PitInTime_sec']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # FreshTyre: 文字列 'True'/'False' → bool
    if 'FreshTyre' in df.columns:
        df['FreshTyre'] = df['FreshTyre'].astype(str).str.strip().str.lower() == 'true'

    print(f'[OK] {gp_label}: {len(df)}行 / {df["Driver"].nunique()}ドライバー')
    return df


# 全GP読み込み
raw_data = {}
for gp_label, csv_path in GP_FILES:
    df = load_race_laps(gp_label, csv_path)
    if df is not None:
        raw_data[gp_label] = df

print(f'\n読み込み完了: {list(raw_data.keys())}')

## Step 2: ロングラン識別ロジック

In [ ]:
def extract_clean_longruns(df, gp_label):
    """
    クリーンなロングランラップを抽出する。

    Returns:
        (clean_df, stats_dict)
    """
    stats = {
        'gp': gp_label,
        'total_laps': len(df),
        'after_green_filter': 0,
        'after_pit_filter': 0,
        'longrun_laps': 0,
        'after_107_filter': 0,
        'excluded_by_107': 0,
        'longrun_count': 0,
        'clean_lap_count': 0,
    }

    # --- Step 6: TrackStatus == '1' グリーンフラグのみ ---
    # 文字列として比較（lessons.md記載: TrackStatusの'1'はstr比較が必要）
    df_green = df[df['TrackStatus'].astype(str).str.strip() == GREEN_FLAG].copy()
    stats['after_green_filter'] = len(df_green)

    # --- Step 1,2: アウトラップ・インラップ除外 ---
    df_clean = df_green[
        df_green['PitOutTime_sec'].isna() &   # アウトラップを除外
        df_green['PitInTime_sec'].isna()       # インラップを除外
    ].copy()
    stats['after_pit_filter'] = len(df_clean)

    # LapTimeがNaNのラップを除外
    df_clean = df_clean.dropna(subset=['LapTime_sec'])

    # --- Step 3,4: 同一ドライバー・Stint・Compoundでグループ化 >= 5周 ---
    group_cols = ['GP', 'Driver', 'Stint', 'Compound']
    group_sizes = df_clean.groupby(group_cols)['LapNumber'].count().reset_index()
    group_sizes.columns = group_cols + ['StintLapCount']

    longrun_groups = group_sizes[group_sizes['StintLapCount'] >= LONGRUN_MIN_LAPS]
    df_longrun = df_clean.merge(longrun_groups[group_cols], on=group_cols, how='inner').copy()
    stats['longrun_laps'] = len(df_longrun)

    # --- Step 5: 107%フィルタ ---
    # セッション全体（グリーン・ピット除外後）の最速ラップを基準にする
    session_fastest = df_clean['LapTime_sec'].min()
    threshold_107 = session_fastest * FILTER_PCT

    df_filtered = df_longrun[df_longrun['LapTime_sec'] <= threshold_107].copy()
    excluded_107 = len(df_longrun) - len(df_filtered)
    stats['after_107_filter'] = len(df_filtered)
    stats['excluded_by_107'] = excluded_107

    if len(df_filtered) == 0:
        print(f'[WARNING] {gp_label}: クリーンなロングランが0件')
        return pd.DataFrame(), stats

    # --- LongRunID付与 ---
    df_filtered = df_filtered.sort_values(['Driver', 'Stint', 'LapNumber']).copy()
    df_filtered['LongRunID'] = (
        df_filtered['GP'] + '_' +
        df_filtered['Driver'].astype(str) + '_S' +
        df_filtered['Stint'].astype(int).astype(str)
    )

    # CleanLapCount: そのLongRunID内の107%フィルタ後ラップ数
    df_filtered['CleanLapCount'] = df_filtered.groupby('LongRunID')['LapNumber'].transform('count')

    stats['longrun_count'] = df_filtered['LongRunID'].nunique()
    stats['clean_lap_count'] = len(df_filtered)

    return df_filtered, stats


# 全GP処理
all_clean_dfs = []
all_stats = []

for gp_label, df in raw_data.items():
    print(f'\n=== {gp_label} ===')
    df_clean, stats = extract_clean_longruns(df, gp_label)
    all_stats.append(stats)
    if len(df_clean) > 0:
        all_clean_dfs.append(df_clean)
        print(f'  → {stats["longrun_count"]}スティント / {stats["clean_lap_count"]}ラップ抽出')

print('\n処理完了')

## Step 3: 統計サマリー

In [ ]:
# 統計サマリーの表示
stats_df = pd.DataFrame(all_stats)
stats_df = stats_df.rename(columns={
    'gp': 'GP',
    'total_laps': '総ラップ数',
    'after_green_filter': 'グリーン後',
    'after_pit_filter': 'ピット除外後',
    'longrun_laps': 'ロングラン候補',
    'excluded_by_107': '107%除外数',
    'clean_lap_count': 'クリーンラップ数',
    'longrun_count': 'ロングランID数',
})

display_cols = ['GP', '総ラップ数', 'グリーン後', 'ピット除外後', 'ロングラン候補',
                '107%除外数', 'クリーンラップ数', 'ロングランID数']
print('【GPごとの統計サマリー】')
print(stats_df[display_cols].to_string(index=False))

## Step 4: CSV出力

In [ ]:
# 出力カラムを仕様に合わせて選択
out_cols = [
    'GP', 'Driver', 'Team', 'Stint', 'Compound',
    'TyreLife', 'LapNumber', 'LapTime_sec',
    'Sector1Time_sec', 'Sector2Time_sec', 'Sector3Time_sec',
    'FreshTyre', 'LongRunID', 'CleanLapCount'
]

if all_clean_dfs:
    df_final = pd.concat(all_clean_dfs, ignore_index=True)
    available_cols = [c for c in out_cols if c in df_final.columns]
    df_final = df_final[available_cols]

    output_csv = os.path.join(OUTPUT_DIR, 'clean_longruns.csv')
    df_final.to_csv(output_csv, index=False, encoding='utf-8')
    print(f'保存: {output_csv}')
    print(f'行数: {len(df_final)}')
    print(f'ドライバー数: {df_final["Driver"].nunique()}')
    print(f'GP数: {df_final["GP"].nunique()}')
    print(f'ロングランID数: {df_final["LongRunID"].nunique()}')
    print('\n先頭5行:')
    print(df_final.head().to_string())
else:
    print('[WARNING] 出力データが空です')
    df_final = pd.DataFrame()

## Step 5: 可視化 — GPごとのロングラン統計

In [ ]:
if len(df_final) > 0:
    fig, axes = plt.subplots(1, 3, figsize=STYLE['figsize'])
    fig.patch.set_facecolor(STYLE['bg_color'])
    fig.suptitle('Long Run Stats by GP (R01-R03)',
                 color=STYLE['text_color'], fontsize=STYLE['title_size'],
                 fontweight='bold', y=1.02)

    gp_labels_list = [s['gp'] for s in all_stats]
    x = list(range(len(gp_labels_list)))

    subplot_data = [
        ('Long Run Stints',   'longrun_count',    '#4488ff'),
        ('Clean Laps',        'clean_lap_count',  '#44cc88'),
        ('Excluded by 107%',  'excluded_by_107',  '#ff6644'),
    ]

    for ax, (title, key, color) in zip(axes, subplot_data):
        ax.set_facecolor(STYLE['bg_color'])
        vals = [s[key] for s in all_stats]
        bars = ax.bar(x, vals, color=color, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(gp_labels_list, color=STYLE['text_color'],
                           fontsize=9, rotation=15, ha='right')
        ax.set_title(title, color=STYLE['text_color'], fontsize=STYLE['label_size'])
        ax.tick_params(colors=STYLE['text_color'])
        ax.spines[['top', 'right']].set_visible(False)
        for spine in ['left', 'bottom']:
            ax.spines[spine].set_color(STYLE['grid_color'])
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.3, str(val),
                    ha='center', va='bottom',
                    color=STYLE['text_color'], fontsize=10)

    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, 'gp_summary.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=STYLE['bg_color'])
    plt.show()
    print(f'保存: {out_path}')

## Step 6: 可視化 — コンパウンド別ロングラン分布

In [ ]:
if len(df_final) > 0:
    # GP×Compoundのクロス集計（ドライバー×スティントでデデュープ）
    pivot = (
        df_final
        .drop_duplicates(subset=['GP', 'Driver', 'Stint', 'Compound'])
        .groupby(['GP', 'Compound'])
        .size()
        .unstack(fill_value=0)
    )

    compound_order = ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']
    ordered_cols = [c for c in compound_order if c in pivot.columns]
    pivot = pivot[ordered_cols]

    fig, ax = plt.subplots(figsize=STYLE['figsize'])
    fig.patch.set_facecolor(STYLE['bg_color'])
    ax.set_facecolor(STYLE['bg_color'])

    bottom = [0] * len(pivot)
    for compound in ordered_cols:
        vals = pivot[compound].values
        color = COMPOUND_COLORS.get(compound, '#888888')
        bars = ax.bar(pivot.index, vals, bottom=bottom, label=compound,
                      color=color, alpha=0.85,
                      edgecolor='#555555', linewidth=0.5)
        for i, (bar, val) in enumerate(zip(bars, vals)):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bottom[i] + val / 2, str(val),
                        ha='center', va='center',
                        color='#000000' if compound == 'MEDIUM' else STYLE['text_color'],
                        fontsize=9, fontweight='bold')
        bottom = [b + v for b, v in zip(bottom, vals)]

    ax.set_title('Compound Distribution of Long Runs (R01-R03)',
                 color=STYLE['text_color'], fontsize=STYLE['title_size'], fontweight='bold')
    ax.set_xlabel('GP', color=STYLE['text_color'], fontsize=STYLE['label_size'])
    ax.set_ylabel('Long Run Stints', color=STYLE['text_color'], fontsize=STYLE['label_size'])
    ax.tick_params(colors=STYLE['text_color'])
    ax.spines[['top', 'right']].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color(STYLE['grid_color'])

    legend = ax.legend(title='Compound', title_fontsize=10,
                       facecolor='#2a2a4e', edgecolor=STYLE['grid_color'],
                       labelcolor=STYLE['text_color'])
    legend.get_title().set_color(STYLE['text_color'])
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, 'compound_distribution.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=STYLE['bg_color'])
    plt.show()
    print(f'保存: {out_path}')

## Step 7: ドライバー×スティント ロングラン一覧テーブル

In [ ]:
if len(df_final) > 0:
    for gp_label in df_final['GP'].unique():
        df_gp = df_final[df_final['GP'] == gp_label]

        # ドライバー・スティントごとにサマリー
        summary = (
            df_gp
            .groupby(['Driver', 'Team', 'Stint', 'Compound'])
            .agg(
                CleanLaps=('LapNumber', 'count'),
                MinLapTime=('LapTime_sec', 'min'),
                MaxLapTime=('LapTime_sec', 'max'),
                AvgLapTime=('LapTime_sec', 'mean'),
                TyreLifeStart=('TyreLife', 'min'),
            )
            .reset_index()
            .sort_values(['Driver', 'Stint'])
        )

        # テーブル描画
        n_rows = len(summary)
        fig_height = max(4, n_rows * 0.35 + 2)
        fig, ax = plt.subplots(figsize=(14, fig_height))
        fig.patch.set_facecolor(STYLE['bg_color'])
        ax.set_facecolor(STYLE['bg_color'])
        ax.axis('off')

        col_labels = ['Driver', 'Team', 'Stint', 'Compound',
                      'Clean\nLaps', 'Min\nLapTime', 'Max\nLapTime',
                      'Avg\nLapTime', 'TyreLife\nStart']

        cell_data = []
        for _, row in summary.iterrows():
            cell_data.append([
                row['Driver'], row['Team'],
                int(row['Stint']), row['Compound'],
                int(row['CleanLaps']),
                f"{row['MinLapTime']:.3f}",
                f"{row['MaxLapTime']:.3f}",
                f"{row['AvgLapTime']:.3f}",
                int(row['TyreLifeStart']),
            ])

        table = ax.table(
            cellText=cell_data,
            colLabels=col_labels,
            cellLoc='center',
            loc='center',
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.0, 1.4)

        for (row_idx, col_idx), cell in table.get_celld().items():
            if row_idx == 0:
                cell.set_facecolor('#2a2a5e')
                cell.set_text_props(color=STYLE['text_color'], fontweight='bold')
            else:
                if col_idx == 3 and row_idx <= len(cell_data):
                    compound_val = cell_data[row_idx - 1][3]
                    cell.set_facecolor(COMPOUND_COLORS.get(compound_val, '#333355'))
                    text_color = '#000000' if compound_val == 'MEDIUM' else STYLE['text_color']
                    cell.set_text_props(color=text_color, fontweight='bold')
                else:
                    bg = '#1e1e38' if row_idx % 2 == 0 else '#15152a'
                    cell.set_facecolor(bg)
                    cell.set_text_props(color=STYLE['text_color'])
            cell.set_edgecolor(STYLE['grid_color'])

        ax.set_title(f'{gp_label} — Driver x Stint Long Run List',
                     color=STYLE['text_color'], fontsize=STYLE['title_size'],
                     fontweight='bold', pad=12)

        plt.tight_layout()
        out_path = os.path.join(OUTPUT_DIR, f'longrun_table_{gp_label}.png')
        plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=STYLE['bg_color'])
        plt.show()
        print(f'保存: {out_path} ({n_rows}スティント)')

## Step 8: クロスGP比較 — ドライバー別ロングランペース

In [ ]:
if len(df_final) > 0:
    # 全GP×ドライバーの中央ラップタイム（ロングランのみ）
    driver_pace = (
        df_final
        .groupby(['GP', 'Driver', 'Team'])
        ['LapTime_sec']
        .median()
        .reset_index()
        .rename(columns={'LapTime_sec': 'MedianLapTime'})
    )

    print('【ドライバー別 中央ロングランラップタイム】')
    for gp in driver_pace['GP'].unique():
        print(f'\n--- {gp} ---')
        sub = driver_pace[driver_pace['GP'] == gp].sort_values('MedianLapTime')
        for _, row in sub.iterrows():
            print(f'  {row["Driver"]:5s} ({row["Team"]:<20s}): {row["MedianLapTime"]:.3f}秒')

In [ ]:
print('\n=== 処理完了 ===')
print(f'出力ディレクトリ: {OUTPUT_DIR}')
print('作成ファイル:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {f} ({size_kb:.1f} KB)')